In [ ]:
import torch, glob, hashlib, shutil
from pathlib import Path
PRIMARY_SHA="12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
FT_SHA="f6bbdfe88c43a4e1addd38b64edcbb7b2050ee3cac4a4db7f08154fa058e985c"
found={}
for c in sorted(set(glob.glob("/kaggle/input/**/edge_predictor_best.pth", recursive=True))):
    h=hashlib.sha256(open(c,"rb").read()).hexdigest(); found[h]=c; print(h[:12], c)
prim=torch.load(found[PRIMARY_SHA], map_location="cpu", weights_only=True)
ft=torch.load(found[FT_SHA], map_location="cpu", weights_only=True)
assert set(prim)==set(ft)
prefixes=sorted({k.split(".")[0] for k in prim}); print("top-level modules:", prefixes)
for p in prefixes:
    ks=[k for k in prim if k.startswith(p+".")]; n=sum(prim[k].numel() for k in ks); print(f"  {p}: {len(ks)} tensors, {n:,} params")
def is_detection(k):
    top=k.split(".")[0].lower()
    return top.startswith("unet") or "det" in top or "heat" in top or "center" in top
hyb={}; n_p=n_f=0
for k in prim:
    if is_detection(k): hyb[k]=prim[k]; n_p+=1
    else: hyb[k]=ft[k]; n_f+=1
print(f"hybrid: {n_p} tensors from PRIMARY (unet/detection), {n_f} from H100 ft (association)")
out=Path("/kaggle/working/hybrid"); out.mkdir(exist_ok=True)
torch.save(hyb, out/"edge_predictor_best.pth")
cfg=glob.glob("/kaggle/input/**/weights/unet_transformer/split_0/config.json", recursive=True)[0]; shutil.copy(cfg, out/"config.json")
print("HYBRID SHA256:", hashlib.sha256(open(out/"edge_predictor_best.pth","rb").read()).hexdigest())
